In [1]:
import pandas as pd

DATA_PATH = "/Users/mayarabin/Desktop/Thesis Downloads V2/ICPSR_09484 4/DS0001/09484-0001-Data.txt"

# ---------------------------------------------------------------------------
# Reference field specs: (column_name, start_1idx, length)
# ---------------------------------------------------------------------------
REFERENCE_FIELDS = [
    ("state_code",    1,   2),
    ("type_of_govt",  3,   1),
    ("county_code",   4,   3),
    ("unit_code",     7,   3),
    ("check_digit",   10,  1),
    ("gid_sector",    11,  2),
    ("suppid",        13,  5),
    ("gov_name",      18,  32),
    ("region_code",   50,  1),
    ("county_name",   51,  30),
    ("fips_code",     81,  11),
    ("pop",           92,  7),
    ("fiscal_yr_end", 116, 4),
    ("year_of_data",  164, 2),
]

# ---------------------------------------------------------------------------
# Financial item specs: (column_name, start_1idx)
# All items are 12 characters wide.
# Positions from 1987 Governments Finance Dictionary (ICPSR 9484, pp. 2-5 to 2-7).
# OCR garbling in codebook corrected: "101"->T01, "714"->T14, "B&l"->B81, etc.
# ---------------------------------------------------------------------------
T_CODES = [
    ("T01", 201),  # Property Tax
    ("T02", 213),  # Parent Govt Contribution - Elem/Sec
    ("T03", 225),  # Property Tax Relief Amount
    ("T04", 237),  # [Subcode] Property Tax - Higher Ed
    ("T05", 249),  # Parent Govt Contribution - Higher Ed
    ("T06", 261),  # [Subcode] Property Tax - Elem/Sec
    ("T08", 273),  # Customs Duties (Federal only)
    ("T09", 285),  # Total General Sales Taxes
    ("T10", 297),  # Alcoholic Beverage Sales Tax
    ("T11", 309),  # Amusement Tax
    ("T12", 321),  # Insurance Premium Tax
    ("T13", 333),  # Motor Fuels Sales Tax
    ("T14", 345),  # Parimutuels Tax
    ("T15", 357),  # Public Utilities Tax
    ("T16", 369),  # Tobacco Sales Tax
    ("T19", 381),  # Other Selective Sales Taxes
    ("T20", 393),  # Alcoholic Beverage License Tax
    ("T21", 405),  # Amusement License Tax
    ("T22", 417),  # Corporation License Tax
    ("T23", 429),  # Hunting & Fishing License
    ("T24", 441),  # Motor Vehicle License
    ("T25", 453),  # Motor Vehicle Operators License
    ("T27", 465),  # Public Utility License Tax
    ("T28", 477),  # Occup & Business License NEC
    ("T29", 489),  # Other License Tax
    ("T40", 501),  # Individual Income Tax
    ("T41", 513),  # Corporation Net Income Tax
    ("T50", 525),  # Death & Gift Tax
    ("T51", 537),  # Documentary & Stock Trans Tax
    ("T53", 549),  # Severance Tax
    ("T99", 561),  # Taxes NEC
]

B_CODES = [
    ("B01", 1017),  # Federal IG Air Transportation
    ("B21", 1029),  # Federal IG Education
    ("B22", 1041),  # Federal IG Employment Security Admin
    ("B23", 1053),  # Rev on Behalf of School - Federal
    ("B25", 1065),  # [Subcode] Dir Fed - Higher Ed
    ("B26", 1077),  # [Subcode] Dir Fed - Elem/Sec
    ("B27", 1089),  # Federal General Revenue Sharing
    ("B30", 1101),  # Federal IG General Support
    ("B42", 1113),  # Federal IG Health & Hospitals
    ("B46", 1125),  # Federal IG Highways
    ("B47", 1137),  # Federal IG Transit Subsidy
    ("B50", 1149),  # Federal IG Housing & Comm Dev
    ("B54", 1161),  # Federal IG Agriculture
    ("B59", 1173),  # Federal IG Other Natural Resources
    ("B79", 1185),  # Federal IG Public Welfare
    ("B81", 1197),  # Federal IG Sewerage
    ("B89", 1209),  # Federal IG Other
]

ALL_ITEMS = T_CODES + B_CODES

# ---------------------------------------------------------------------------
# Parser
# ---------------------------------------------------------------------------
def parse_financial_field(s):
    s = s.strip()
    if not s:
        return 0
    try:
        return int(s)
    except ValueError:
        return 0

def parse_record(line):
    row = {}
    for name, start, length in REFERENCE_FIELDS:
        row[name] = line[start - 1 : start - 1 + length].strip()
    for name, start in ALL_ITEMS:
        row[name] = parse_financial_field(line[start - 1 : start - 1 + 12])
    return row

records = []
with open(DATA_PATH, "r", encoding="ascii", errors="replace") as f:
    for i, line in enumerate(f):
        if len(line.strip()) == 0:
            continue
        if len(line.rstrip("\n")) < 200:
            print(f"Warning: short line at row {i}, skipping")
            continue
        records.append(parse_record(line))

df_1987 = pd.DataFrame(records)
df_1987["pop"] = pd.to_numeric(df_1987["pop"], errors="coerce")
df_1987["year_of_data"] = pd.to_numeric(df_1987["year_of_data"], errors="coerce")

print(f"Loaded {len(df_1987):,} records")
print(f"\nType of govt breakdown:\n{df_1987['type_of_govt'].value_counts().sort_index()}")
print(f"\nSample municipalities:")
print(df_1987[df_1987["type_of_govt"] == "2"].head(5)[["gov_name","state_code","fips_code","pop","T01","B27","B89"]])

Loaded 83,835 records

Type of govt breakdown:
type_of_govt
0       50
1     3042
2    19217
3    16695
4    29427
5    15403
6        1
Name: count, dtype: int64

Sample municipalities:
                gov_name state_code    fips_code    pop  T01  B27  B89
69  TOWN OF AUTAUGAVILLE         01  01001524000    901    0    2    0
70      BILLINGSLEY TOWN         01  01001524000    110    1    0    0
71       PRATTVILLE CITY         01  01001524000  21122  290  318    0
72      BAY MINETTE CITY         01  01003516000   7765  248  238    0
73        CITY OF DAPHNE         01  01003516000   4177  161   50    0


In [2]:
# Filter to municipalities (type 2) and townships (type 3)
df_1987_muni = df_1987[df_1987["type_of_govt"].isin(["2", "3"])].copy()
print(f"Filtered to {len(df_1987_muni):,} records ({len(df_1987[df_1987['type_of_govt']=='2']):,} municipalities, {len(df_1987[df_1987['type_of_govt']=='3']):,} townships)")



Filtered to 35,912 records (19,217 municipalities, 16,695 townships)


In [5]:
# Limits down to b and t column codes and aggregates by municipality
b_cols = [c for c in df_1987_muni.columns if c.startswith("B")]
t_cols = [c for c in df_1987_muni.columns if c.startswith("T")]

id_cols = ["gov_name", "state_code", "county_code", "unit_code", "fips_code", "pop", "year_of_data", "type_of_govt"]

b_rows = df_1987_muni[id_cols].copy()
b_rows["revenue_type"] = "B"
b_rows["total_revenue"] = df_1987_muni[b_cols].sum(axis=1)

t_rows = df_1987_muni[id_cols].copy()
t_rows["revenue_type"] = "T"
t_rows["total_revenue"] = df_1987_muni[t_cols].sum(axis=1)

df_1987_long = pd.concat([b_rows, t_rows], ignore_index=True).sort_values(["state_code", "unit_code", "revenue_type"]).reset_index(drop=True)

print(f"Rows: {len(df_1987_long):,} ({len(df_1987_muni):,} munis x 2)")
print(f"\nSample:")
print(df_1987_long[df_1987_long["gov_name"].str.contains("PRATTVILLE")].to_string())

Rows: 71,824 (35,912 munis x 2)

Sample:
            gov_name state_code county_code unit_code    fips_code    pop  year_of_data type_of_govt revenue_type  total_revenue
260  PRATTVILLE CITY         01         001       003  01001524000  21122          87.0            2            B            318
312  PRATTVILLE CITY         01         001       003  01001524000  21122          87.0            2            T           4040
